In [1]:
import cv2
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ---------------- CONFIG ----------------
DATA_ROOT = Path(
    "SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions"
) / "SnowPole_Detection_Dataset"

MODALITIES = ["reflec", "signal", "nearir", "range"]
SPLIT = "train"
MAX_PIXELS = 50000
# ---------------------------------------


def read_first_channel(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(path)

    # If multi-channel, take only first
    if img.ndim == 3:
        img = img[:, :, 0]

    return img.astype(np.float32)

def normalize01(img):
    img = img.astype(np.float32)
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-6)


def edge_residual(range_img, reflec_img):
    r = normalize01(range_img)
    f = normalize01(reflec_img)

    rx = cv2.Sobel(r, cv2.CV_32F, 1, 0, ksize=3)
    ry = cv2.Sobel(r, cv2.CV_32F, 0, 1, ksize=3)
    fx = cv2.Sobel(f, cv2.CV_32F, 1, 0, ksize=3)
    fy = cv2.Sobel(f, cv2.CV_32F, 0, 1, ksize=3)

    grad_r = np.sqrt(rx**2 + ry**2)
    grad_f = np.sqrt(fx**2 + fy**2)

    edge = 0.5 * grad_r + 0.5 * grad_f
    return normalize01(edge)


def collect_samples(data_root, modalities, split, max_pixels):
    modality_dirs = {
        m: data_root / m / split for m in modalities
    }

    ref_files = sorted(modality_dirs["reflec"].glob("*.png"))
    samples = []

    for ref_path in ref_files:
        imgs = []
        for m in modalities:
            img_path = modality_dirs[m] / ref_path.name
            if not img_path.exists():
                raise FileNotFoundError(f"Missing {img_path}")

            img = read_first_channel(img_path)
            imgs.append(img)

        stacked = np.stack(imgs, axis=-1).reshape(-1, 4)
        samples.append(stacked)

    samples = np.concatenate(samples, axis=0)

    if len(samples) > max_pixels:
        idx = np.random.choice(len(samples), max_pixels, replace=False)
        samples = samples[idx]

    return samples


# ---- Collect data ----
X = collect_samples(DATA_ROOT, MODALITIES, SPLIT, MAX_PIXELS)

# ---- Standardize per modality ----
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# ---- PCA ----
pca = PCA(n_components=4)
pca.fit(X_std)

# ---- Modality contribution ----
contrib = np.sum(
    np.abs(pca.components_) * pca.explained_variance_[:, None],
    axis=0
)

weights = contrib / contrib.sum()

# ---- Output ----
print("PCA-derived modality weights:")
for m, w in zip(MODALITIES, weights):
    print(f"{m:12s}: {w:.4f}")


PCA-derived modality weights:
reflec      : 0.2485
signal      : 0.2555
nearir      : 0.2320
range       : 0.2640


In [2]:
import os
import cv2
import numpy as np
from pathlib import Path

# ---------------- CONFIG ----------------
ROOT = Path(
    "SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions"
) / "SnowPole_Detection_Dataset"
OUT_ROOT = Path("dataset_fused_residual/images")

MODALITIES = {
    "reflec": 0.2485,
    "signal": 0.2555,
    "nearir": 0.2320,
    "range":  0.2640
}

SPLITS = ["train", "valid", "test"]
IMG_EXTS = [".png", ".jpg", ".jpeg"]

# ----------------------------------------

# Normalize weights
weights = np.array(list(MODALITIES.values()), dtype=np.float32)
weights = weights / weights.sum()
modalities = list(MODALITIES.keys())

def normalize(img):
    img = img.astype(np.float32)
    mean = img.mean()
    std = img.std() + 1e-6
    return (img - mean) / std

for split in SPLITS:
    out_dir = OUT_ROOT / split
    out_dir.mkdir(parents=True, exist_ok=True)

    refl_dir = ROOT / modalities[0] / split
    images = [p for p in refl_dir.iterdir() if p.suffix.lower() in IMG_EXTS]

    for img_path in images:
        imgs = {}

        # Read and normalize all modalities
        for mod in modalities:
            mod_path = ROOT / mod / split / img_path.name
            if not mod_path.exists():
                raise FileNotFoundError(f"Missing: {mod_path}")

            img = cv2.imread(str(mod_path), cv2.IMREAD_GRAYSCALE)
            imgs[mod] = normalize(img)

        # PCA-weighted fusion (low-frequency)
        fused = np.zeros_like(imgs[modalities[0]], dtype=np.float32)
        for w, mod in zip(weights, modalities):
            fused += w * imgs[mod]

        # Edge residual (high-frequency)
        E = edge_residual(
            range_img=imgs["range"],
            reflec_img=imgs["reflec"]
        )

        gamma = 0.05  # try 0.05 first, then 0.1 if needed
        fused = fused + gamma * E


        # Rescale to uint8
        fused = fused - fused.min()
        fused = fused / (fused.max() + 1e-6)
        fused = (fused * 255).astype(np.uint8)

        cv2.imwrite(str(out_dir / img_path.name), fused)

print("Weighted stacking completed.")


Weighted stacking completed.


In [3]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="weighted_fusion.yaml",
    imgsz=1024,
    epochs=500,
    patience=40,        # early stopping
    batch=8,
    device=0,
    project="weighted_late_gate_experiments_residual",
    name="weighted_yolo11n_late_gate_residual",
    amp=False,
    augment=False,
    workers=0,
)

Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=weighted_fusion.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000184AAB26590>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480